In [ ]:
# %% REDItools A-to-I RNA Editing Analysis

In [ ]:
# %% Setup and Installation (run in terminal, not jupyter)
# bash
# Install REDItools (run this in terminal, not jupyter)
# Create conda environment
# conda create -n reditools python=3.8
# conda activate reditools
# Install dependencies
# conda install -c bioconda samtools tabix htslib
# pip install REDItools2

In [7]:
# %% Import Libraries
import os
import pandas as pd
import numpy as np
import glob
from pathlib import Path

# Set display options (like R's options())
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [11]:
# %% Define Paths
# Input paths
bam_dir = '/orange/mingyi.xie/hkates/A001-snaR/aim-1/A-to-I-editing/aligned/GSE99249_ucsc/'
ref_genome = '/orange/cancercenter-dept/GENOMES/iGenomes/references/Homo_sapiens/Ensembl/GRCh38/Sequence/WholeGenomeFasta/ucscHg38Genome.fa'
snar_regions = '/blue/mingyi.xie/hkates/A001-snaR/aim-1/A-to-I-editing/AnnotationAndRegions/snaR-A_hg38.refseq_format.bed.gz'

# Output directory (CHANGE THIS to where you want results)
output_dir = '/blue/mingyi.xie/hkates/A001-snaR/aim-1/A-to-I-editing/REDItools/outs'
os.makedirs(output_dir, exist_ok=True)

In [17]:
# %% Load snaR-A Region Coordinates
# Read only the first 6 columns (standard BED format)
snar_df = pd.read_csv(
    snar_regions, 
    sep='\t', 
    header=None,
    usecols=[0, 1, 2, 3, 4, 5],  # Only read first 6 columns
    names=['chr', 'start', 'end', 'name', 'score', 'strand']
)

print(f"Loaded {len(snar_df)} snaR-A regions")
print(snar_df.head())
print(snar_df.dtypes)

# Create position list for filtering
snar_positions = []
for idx, row in snar_df.iterrows():
    snar_positions.extend(range(row['start'], row['end'] + 1))
snar_positions = set(snar_positions)

print(f"Total positions in snaR-A regions: {len(snar_positions)}")

Loaded 14 snaR-A regions
     chr     start       end      name     score strand
0  chr19  53102747  53102864  snaR-A_1  snaR-A_1      +
1  chr19  53113498  53113615  snaR-A_2  snaR-A_2      +
2  chr19  53118848  53118965  snaR-A_3  snaR-A_3      +
3  chr19  53129250  53129367  snaR-A_4  snaR-A_4      +
4  chr19  53139991  53140108  snaR-A_5  snaR-A_5      +
chr       object
start      int64
end        int64
name      object
score     object
strand    object
dtype: object
Total positions in snaR-A regions: 1652


In [ ]:
# %% Run REDItools on All BAM Files (bash script - run in terminal)
# #!/bin/bash
# BAM_DIR="/orange/mingyi.xie/hkates/A001-snaR/aim-1/A-to-I-editing/aligned/GSE99249_ucsc/"
# REF="/orange/cancercenter-dept/GENOMES/iGenomes/references/Homo_sapiens/Ensembl/GRCh38/Sequence/WholeGenomeFasta/ucscHg38Genome.fa"
# OUT_DIR="/path/to/your/output/reditools_results/"
# mkdir -p $OUT_DIR
# 
# for bam in ${BAM_DIR}*.bam; do
#     filename=$(basename "$bam" .bam)
#     echo "Processing $filename..."
#     python /path/to/reditools2.0/src/cineca/reditools.py \
#         -f "$bam" \
#         -r "$REF" \
#         -o "${OUT_DIR}${filename}.RES.tsv"
# done

In [ ]:
# %% Load and Process REDItools Results
# Get all REDItools output files
reditools_files = glob.glob(f'{output_dir}/reditools_results/*.RES.tsv')
print(f"Found {len(reditools_files)} REDItools output files")

In [ ]:
# %% Define processing function
# Function to process one REDItools file (similar to an R function)
def process_reditools_file(filepath):
    # Read file
    df = pd.read_csv(filepath, sep='\t')
    
    # Extract sample name from filename
    sample_name = Path(filepath).stem.replace('.RES', '')
    df['Sample'] = sample_name
    
    # Keep only A and T positions (sites where A-to-I editing can occur)
    df = df[df['Reference'].isin(['A', 'T'])]
    
    # Parse base counts from string like "[1,2,3,4]"
    # This is like str_split in R
    base_counts = df['BaseCount[A,C,G,T]'].str.replace('[', '').str.replace(']', '').str.split(',', expand=True)
    df[['count_A', 'count_C', 'count_G', 'count_T']] = base_counts.apply(pd.to_numeric)
    
    # Calculate editing level
    # np.where is like ifelse in R
    df['editing_level'] = np.where(
        df['Reference'] == 'A',
        df['count_G'] / (df['count_G'] + df['count_A']),  # A-to-G editing
        df['count_C'] / (df['count_C'] + df['count_T'])   # T-to-C editing (antisense)
    )
    df['editing_level_pct'] = df['editing_level'] * 100
    
    return df

In [ ]:
# %% Process all files
# Process all files and combine (like rbind in R)
all_results = []
for f in reditools_files:
    print(f"Processing {Path(f).name}...")
    df = process_reditools_file(f)
    all_results.append(df)

df_all = pd.concat(all_results, ignore_index=True)
print(f"Total positions analyzed: {len(df_all)}")

In [ ]:
# %% Filter to snaR-A Regions
# Filter to only snaR-A positions
df_snar = df_all[df_all['Position'].isin(snar_positions)].copy()
print(f"Positions in snaR-A regions: {len(df_snar)}")
print(f"Samples: {df_snar['Sample'].nunique()}")

# Filter by quality (like subset in R)
min_coverage = 10
min_editing = 1.0  # percent

df_snar_filtered = df_snar[
    (df_snar['Coverage-q30'] >= min_coverage) &
    (df_snar['editing_level_pct'] >= min_editing)
]
print(f"After filtering (coverage>={min_coverage}, editing>={min_editing}%): {len(df_snar_filtered)}")

In [ ]:
# %% Calculate Per-Sample Editing Index
# Similar to dplyr::group_by %>% summarize in R
editing_index = df_snar_filtered.groupby('Sample').agg({
    'count_A': 'sum',
    'count_G': 'sum',
    'count_C': 'sum',
    'count_T': 'sum',
    'Position': 'count'  # number of edited sites
}).reset_index()

# Calculate overall editing index for each sample
editing_index['AEI'] = (
    (editing_index['count_G'] + editing_index['count_C']) /
    (editing_index['count_G'] + editing_index['count_A'] +
     editing_index['count_C'] + editing_index['count_T'])
) * 100

editing_index.rename(columns={'Position': 'n_sites'}, inplace=True)
print(editing_index)

In [ ]:
# %% Save Results
# Save to CSV (like write.csv in R)
df_snar_filtered.to_csv(f'{output_dir}/snaR_editing_sites.csv', index=False)
editing_index.to_csv(f'{output_dir}/snaR_editing_index.csv', index=False)
print("Results saved!")

In [ ]:
# %% Summary Statistics
# Summary by sample (like summary() in R)
print("\nEditing Index Summary:")
print(editing_index['AEI'].describe())

In [ ]:
# %% Visualize
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(editing_index['AEI'], bins=20, edgecolor='black')
plt.xlabel('A-to-I Editing Index (%)')
plt.ylabel('Number of Samples')
plt.title('Distribution of snaR-A Editing Index')
plt.savefig(f'{output_dir}/editing_index_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# %% Export for R Analysis
# If you prefer to analyze in R, export the data
# Then in R you can: df <- read.csv("snaR_editing_sites.csv")